# HCC1395 兩技術資料：粗拓撲、基因與藥物註解驗證

**TL;DR**：這個 notebook 讀取本輪 frozen outputs，重做關鍵守恆與判定。HCC1395 vs HCC1395_DORADO 的 exact-coordinate 五類 agreement 為 69.39%、κ=0.497，顯著高於染色體內 null；但 aggregate composition、exact tree-set 與 gene/drug matched-null 都不支持『完全一致』或『方法已證明有效』。

## Context & Methods

分析範圍為 chr1–22、7 dataset rows 的 historical layered-v2 engineering snapshot。HCC1395 與 HCC1395_DORADO 是同一 biological HCC1395 的兩個 basecalling/processing rows。五類是：無 HP 內關係、姐妹 only、直系 only、姐妹＋直系、Topo>1 未定。基因主口徑為 GENCODE v46 gene body；COSMIC CGC v104、DGIdb 與 COSMIC CLP HCC1395 只作 context，不是 topology truth。

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
TOPIC = ROOT / 'research/20260712_hcc1395_pair_coarse_topology_gene_drug_validation'
assert TOPIC.exists(), f'Run from InterSubMod repository root: {ROOT}'
topology = json.loads((TOPIC / 'data/topology_pair_analysis.json').read_text())
annotation = pd.read_csv(TOPIC / 'data/hcc1395_annotation_reproducibility.tsv', sep='\t')
summary = pd.read_csv(TOPIC / 'data/coarse_topology_all_dataset_summary.tsv', sep='\t')
status = json.loads((TOPIC / 'data/latest_pipeline_status.json').read_text())
print('Loaded:', len(summary), 'dataset rows;', len(annotation), 'annotation strata')

## Data checks

In [ ]:
classes = ['no_within_hp_relation', 'sister_only', 'direct_only', 'sister_and_direct', 'topology_multiple_unresolved']
assert topology['validation'] == {'checks': 64, 'passed': 64}
assert len(summary) == 7
assert (summary[classes].sum(axis=1) == summary['complete_regions']).all()
assert ((summary['complete_regions'] + summary['incomplete_regions']) == summary['primary_regions']).all()
exact = next(row for row in topology['hcc1395_pair_metrics'] if row['scenario'] == 'exact_coordinate')
assert exact['matched_all'] == 6252 and exact['complete_both'] == 5720
assert abs(exact['raw_agreement'] - 3969/5720) < 1e-15
assert int(annotation.loc[annotation.feature == 'ALL', 'n_present'].iloc[0]) == 5720
print('PASS: 64/64 topology checks; five classes conserve complete; exact baseline reproduces 3969/5720')

## Results

In [ ]:
display(summary[['sample', 'primary_regions', 'complete_regions', *classes]])
pd.DataFrame([{
    'exact matched': exact['matched_all'],
    'complete both': exact['complete_both'],
    'five-class agreement': exact['raw_agreement'],
    'Cohen kappa': exact['cohen_kappa'],
    'null agreement mean': exact['permutation_null']['agreement_mean'],
    'permutation p': exact['permutation_null']['agreement_p_ge'],
    'resolved/unresolved agreement': exact['resolution_binary_agreement'],
    'ordered exact tree-set digest agreement': exact['ordered_exact_tree_set_digest_agreement'],
}])

In [ ]:
annotation_view = annotation.loc[annotation.feature != 'ALL', [
    'feature_label', 'n_present', 'present_agreement', 'present_ci95_low', 'present_ci95_high',
    'difference_present_minus_absent_pp', 'permutation_null_mean_pp', 'permutation_p_two_sided'
]]
display(annotation_view)
assert (annotation_view['permutation_p_two_sided'] > 0.05).all()
print('No annotation stratum shows evidence of extra topology reproducibility at alpha=0.05.')

In [ ]:
hcc = summary.set_index('sample').loc['HCC1395']
dorado = summary.set_index('sample').loc['HCC1395_DORADO']
tv = 0.5 * sum(abs(hcc[f'share_{c}'] - dorado[f'share_{c}']) for c in classes)
verdict = {
    'interval_reproducibility': 'qualified support',
    'coarse_topology_reproducibility': 'partial / moderate',
    'marginal_total_variation': tv,
    'biological_validity': 'not established',
    'method_effectiveness_proof': 'NO-GO',
    'clean_v3_release_ready': False,
}
verdict

## Takeaways & limitations

1. 區間與五類拓撲都有高於 null 的 technical repeatability，但不是完全一致；主要差異來自 resolved vs Topo>1。
2. 兩邊都 resolved 後的 87.18% 是條件式一致，不能取代全體 69.39%。
3. CGC／DGIdb／COSMIC CLP strata 沒有通過 matched-null；它們提供可解釋的生物背景，不提供 topology accuracy 或藥物 actionability。
4. historical layered-v2 有 upstream mismatch；clean-v3 canonical/sensitivity verification 未完成。
5. 真正 efficacy proof 仍需獨立 biological replicates、single-cell／multi-region clone truth 或其他正交驗證。